In [9]:
# Cell 1 — Environment & memory info
import os
import sys
import numpy as np
import platform

print(f"Python version: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Working directory: {os.getcwd()}")

# Optional memory check (install psutil if missing: pip install psutil)
try:
    import psutil
    mem = psutil.virtual_memory()
    print(f"Available memory: {mem.available / (1024 ** 3):.2f} GB")
    print(f"Total memory: {mem.total / (1024 ** 3):.2f} GB")
except ImportError:
    print("psutil not available, skipping memory check")


Python version: 3.11.13 (main, Jun  5 2025, 08:21:08) [Clang 14.0.6 ]
Platform: macOS-15.6-arm64-arm-64bit
Working directory: /Users/lathurzan/Desktop/assignments/AdvanceProgramming/AdvanceProgrammingML
Available memory: 3.51 GB
Total memory: 16.00 GB


In [10]:
# Cell 2 — Import TF and attempt to enable GPU memory growth if GPU exists
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"Found {len(gpus)} GPU(s): {gpus}")
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Enabled memory growth on GPUs")
    except Exception as e:
        print("Could not set memory growth:", e)
else:
    print("No GPU found — running on CPU")


TensorFlow version: 2.19.1
No GPU found — running on CPU


In [11]:
# Cell 3 — Dataset path and parameters
dataset_path = "all_images"   # <-- your folder with subfolders bedbug/, infection/, rash/
if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Dataset folder not found: {dataset_path}")
else:
    print("Dataset folder found:", dataset_path)

# Image params (adjust if needed)
img_height, img_width = 180, 180
batch_size = 32


Dataset folder found: all_images


In [12]:
# Cell 4 — Load datasets (creates training and validation split from directory)
train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)


Found 105 files belonging to 3 classes.
Using 84 files for training.
Found 105 files belonging to 3 classes.
Using 21 files for validation.


In [13]:
# Cell 5 — Show classes and prepare prefetch/cache for performance
class_names = train_ds.class_names
print("Classes found:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)


Classes found: ['bedbug', 'infection', 'rash']


In [14]:
# Cell 6 — Build a simple CNN model
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(class_names))  # logits output
])

model.summary()


/Users/lathurzan/opt/anaconda3/envs/tf_env/lib/python3.11/site-packages/keras/src/layers/preprocessing/tf_data_layer.py:19: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 180, 180, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 178, 178, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 89, 89, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 87, 87, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 43, 43, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 41, 41, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 20, 20, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 51200)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     6,553,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,647,363 (25.36 MB)

 Trainable params: 6,647,363 (25.36 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
# Cell 7 — Compile
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)
print("Model compiled.")


Model compiled.


In [ ]:
# Cell 8 — Train the model
epochs = 10  # increase if resources allow
print("Training started...")
history = model.fit(train_ds, validation_data=val_ds, epochs=epochs)
print("Training finished.")


Training started...
Epoch 1/10


In [3]:
# Cell 9 — Save the trained model
# Keras native format recommended (.keras), but .h5 also works
model.save("skin_condition_classifier_model.keras")
print("Model saved as 'skin_condition_classifier_model.keras'")


NameError: name 'model' is not defined

In [4]:
# Cell 10 — Predict a single image (adjust path to an actual file)
from tensorflow.keras.preprocessing import image
import numpy as np

test_img_path = "test_images/test1.jpeg"  # change to your test image path
if not os.path.exists(test_img_path):
    print("Test image not found:", test_img_path)
else:
    img = image.load_img(test_img_path, target_size=(img_height, img_width))
    img_arr = image.img_to_array(img)
    img_arr = np.expand_dims(img_arr, axis=0) / 255.0
    logits = model.predict(img_arr)
    probs = tf.nn.softmax(logits[0]).numpy()
    pred_idx = np.argmax(probs)
    print(f"Predicted: {class_names[pred_idx]}  — confidence: {probs[pred_idx]*100:.2f}%")


/Users/lathurzan/opt/anaconda3/envs/tf_env/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


NameError: name 'os' is not defined